# Imports e caminho

In [0]:
import pyspark.sql.functions as F

In [0]:
volume_path_raw_inputs = "/Volumes/cinedata/bronze/raw_inputs"

# Criando tabelas

In [0]:
path_movies_info = f"{volume_path_raw_inputs}/movies_info_TMDB_IMDB.csv"
path_movie_financials = f"{volume_path_raw_inputs}/movies_financials_IMDB_TMDB.csv"
path_movie_metrics = f"{volume_path_raw_inputs}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{volume_path_raw_inputs}/credits_and_tags_IMDB_TMDB.csv"
path_movie_reviews = f"{volume_path_raw_inputs}/movies_reviews.csv"


In [0]:
# Mapeamento dos caminhos dos arquivos para os nomes das tabelas de destino na camada Bronze
mapeamento_tabelas = [
    (path_movies_info, "cinedata.bronze.tb_movies_info"),
    (path_movie_financials, "cinedata.bronze.tb_movies_financials"),
    (path_movie_metrics, "cinedata.bronze.tb_movies_metrics"),
    (path_credits_and_tags, "cinedata.bronze.tb_credits_and_tags"),
    (path_movie_reviews, "cinedata.bronze.tb_movies_reviews")
]

In [0]:


# Loop para processar e salvar cada arquivo
for caminho_csv, nome_tabela in mapeamento_tabelas:
    
    # Ler o CSV
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("escape", '"')
        .load(caminho_csv)
    )
    
    # Adiciona o Timestamp
    df_bronze = df.withColumn("ingestion_datetime", F.current_timestamp())
    
    # Formato Delta utilizando o modo Append
    (
        df_bronze.write.format("delta")
        .mode("append")
        .saveAsTable(nome_tabela)
    )
    
    print(f"Sucesso: O arquivo '{caminho_csv.split('/')[-1]}' foi ingerido na tabela '{nome_tabela}'.")

Sucesso: O arquivo 'movies_info_TMDB_IMDB.csv' foi ingerido na tabela 'cinedata.bronze.tb_movies_info'.
Sucesso: O arquivo 'movies_financials_IMDB_TMDB.csv' foi ingerido na tabela 'cinedata.bronze.tb_movies_financials'.
Sucesso: O arquivo 'movies_metrics_IMDB_TMDB.csv' foi ingerido na tabela 'cinedata.bronze.tb_movies_metrics'.
Sucesso: O arquivo 'credits_and_tags_IMDB_TMDB.csv' foi ingerido na tabela 'cinedata.bronze.tb_credits_and_tags'.
Sucesso: O arquivo 'movies_reviews.csv' foi ingerido na tabela 'cinedata.bronze.tb_movies_reviews'.


In [0]:
display(spark.table(f"cinedata.bronze.tb_movies_info").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-06T19:08:37.556Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-06T19:08:37.556Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-06T19:08:37.556Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-06T19:08:37.556Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-06T19:08:37.556Z


# API 

## Obtendo tempo

In [0]:
from datetime import datetime, timedelta
import requests

In [0]:
# Obter data de hoje e de 7 dias atrás 
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)

In [0]:
# Formatar as datas no padrão da API: MM-DD-AAAA
data_fim_sugerida = hoje.strftime("%m-%d-%Y")
data_inicio_sugerida = sete_dias_atras.strftime("%m-%d-%Y")

In [0]:
# Criar os widgets 
dbutils.widgets.text("data_inicial", data_inicio_sugerida, "Data Inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_final", data_fim_sugerida, "Data Final (MM-DD-AAAA)")

In [0]:
# Capturar os valores que estão nos widgets para usar na URL
data_inicio_formatada = dbutils.widgets.get("data_inicial")
data_fim_formatada = dbutils.widgets.get("data_final")

print(f"Período configurado: {data_inicio_formatada} a {data_fim_formatada}")

Período configurado: 09-29-2026 a 10-06-2026


## Chamada

In [0]:
# Montar o URL do endpoint com os parâmetros de data do widget
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

In [0]:
resposta = requests.get(url)
dados_json = resposta.json()

# Os dados da cotação vêm dentro da chave 'value'
registos_cotacao = dados_json.get("value", [])

In [0]:
dados_json

{'@odata.context': 'https://was-p.bcnet.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata$metadata#_CotacaoDolarPeriodo(cotacaoCompra,dataHoraCotacao)',
 'value': [{'cotacaoCompra': 5.2198,
   'dataHoraCotacao': '2026-09-29 13:06:06.446305'},
  {'cotacaoCompra': 5.1803, 'dataHoraCotacao': '2026-09-30 13:11:44.783262'},
  {'cotacaoCompra': 5.2073, 'dataHoraCotacao': '2026-10-01 13:10:35.4469'},
  {'cotacaoCompra': 5.2232, 'dataHoraCotacao': '2026-10-02 13:03:16.256632'},
  {'cotacaoCompra': 4.9853, 'dataHoraCotacao': '2026-10-05 13:07:36.558602'},
  {'cotacaoCompra': 4.9692, 'dataHoraCotacao': '2026-10-06 13:03:21.267287'}]}

In [0]:
registos_cotacao

[{'cotacaoCompra': 5.2198, 'dataHoraCotacao': '2026-09-29 13:06:06.446305'},
 {'cotacaoCompra': 5.1803, 'dataHoraCotacao': '2026-09-30 13:11:44.783262'},
 {'cotacaoCompra': 5.2073, 'dataHoraCotacao': '2026-10-01 13:10:35.4469'},
 {'cotacaoCompra': 5.2232, 'dataHoraCotacao': '2026-10-02 13:03:16.256632'},
 {'cotacaoCompra': 4.9853, 'dataHoraCotacao': '2026-10-05 13:07:36.558602'},
 {'cotacaoCompra': 4.9692, 'dataHoraCotacao': '2026-10-06 13:03:21.267287'}]

In [0]:

# Converter a lista de dicionários para um DataFrame do PySpark
if registos_cotacao:
    df_cotacao = spark.createDataFrame(registos_cotacao)
    display(df_cotacao)
else:
    print("Não foram encontrados registos de cotação para o período selecionado.")

cotacaoCompra,dataHoraCotacao
5.2198,2026-09-29 13:06:06.446305
5.1803,2026-09-30 13:11:44.783262
5.2073,2026-10-01 13:10:35.4469
5.2232,2026-10-02 13:03:16.256632
4.9853,2026-10-05 13:07:36.558602
4.9692,2026-10-06 13:03:21.267287


In [0]:
df_cotacao_bronze = df_cotacao.select("*", F.current_timestamp().alias("ingestion_datetime"))

In [0]:
display(df_cotacao_bronze)

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.2198,2026-09-29 13:06:06.446305,2026-10-06T20:04:54.025Z
5.1803,2026-09-30 13:11:44.783262,2026-10-06T20:04:54.025Z
5.2073,2026-10-01 13:10:35.4469,2026-10-06T20:04:54.025Z
5.2232,2026-10-02 13:03:16.256632,2026-10-06T20:04:54.025Z
4.9853,2026-10-05 13:07:36.558602,2026-10-06T20:04:54.025Z
4.9692,2026-10-06 13:03:21.267287,2026-10-06T20:04:54.025Z


In [0]:
# Gravar a tabela
(
    df_cotacao_bronze.write.format("delta")
    .mode("append")
    .saveAsTable("cinedata.bronze.tb_cotacao_dolar")
)

print("Tabela 'cinedata.bronze.tb_cotacao_dolar' gravada com sucesso!")

Tabela 'cinedata.bronze.tb_cotacao_dolar' gravada com sucesso!
